# ELECTRE II
---
_**Él**imination **E**t **C**hoix **T**raduisant la **Re**alité **II**_

In [1]:
import numpy as np
import pandas as pd

from scipy.sparse.csgraph import connected_components

## 1. Introduction
---
**ELECTRE II** ranks alternatives through **strong** and **weak outranking relations**. It extends the choice approach illustrated in [ELECTRE I](./electre_I.ipynb) to an ordering problem.

This notebook follows the **two concordance levels and criterion vetoes** formulation in [Figueira, Mousseau and Roy (2005), section 3.2.1](https://www.lamsade.dauphine.fr/mcda/biblio/PDF/JFVMBR2005.pdf). Two complete preorders, constructed in opposite directions, are intersected. The result may preserve **ties** or **incomparability**.

### 1.1. Applications
---
The _ELECTRE II method_ supports prioritization with tradeoffs and limits on acceptable disadvantages, such as:

- Road rehabilitation planning
- Public service modernization
- Environmental investment prioritization
- Technology assessment

Consider a municipality prioritizing **5 road corridors** for rehabilitation. Evaluate **annual injury crashes avoided**, **pavement condition index** (PCI, 0–100), **daily users**, and **capital cost** (BRL million). Lower PCI indicates poorer condition and **greater rehabilitation urgency**, so it is minimized here; crashes avoided and users are maximized.

The data, estimated benefits, corridor names and policy parameters are **illustrative**. All projects are feasible. The objective is an order of priority, rather than a portfolio constrained by a fixed budget.

In [2]:
df = pd.DataFrame({
    "crashes avoided": [16, 18, 14, 15, 12],
    "pavement PCI": [10, 30, 40, 50, 60],
    "daily users": [45000, 35000, 30000, 25000, 20000],
    "cost": [8, 9, 6, 7, 5]
})

df.index = [
    "North Avenue",
    "School Corridor",
    "Industrial Link",
    "Market Street",
    "South Connector"
]

df.style.background_gradient()

,crashes avoided,pavement PCI,daily users,cost
North Avenue,16,10,45000,8
School Corridor,18,30,35000,9
Industrial Link,14,40,30000,6
Market Street,15,50,25000,7
South Connector,12,60,20000,5


## 2. Method steps
---

### 2.1. Objectives, weights and policy parameters
---
Define **weights** as the voting power of criteria in a supporting coalition. The team requires **75%** support for a strong assertion and **65%** for a weak assertion. These are policy judgments, not universal constants.

Here, the team also permits a disadvantage of at most **35% of each planning range**, under either relation. For cost, this means **BRL 3.5 million**. The following fixed bounds define these ranges before comparing projects.

In [3]:
OBJECTIVES = pd.Series(["max", "min", "max", "min"], index=df.columns)
WEIGHTS = pd.Series([0.30, 0.25, 0.25, 0.20], index=df.columns)

BOUNDS = pd.DataFrame({
    "lower": [0, 0, 0, 2],
    "upper": [20, 100, 50000, 12]
}, index=df.columns)

C_STRONG = 0.75
C_WEAK = 0.65
VETO = pd.Series(0.35, index=df.columns)
TOLERANCE = 1e-12

assert (WEIGHTS > 0).all() and np.isclose(WEIGHTS.sum(), 1)
assert 0.5 <= C_WEAK < C_STRONG <= 1 - WEIGHTS.min()

pd.DataFrame({
    "objective": OBJECTIVES,
    "weight": WEIGHTS,
    "lower": BOUNDS["lower"],
    "upper": BOUNDS["upper"],
    "veto in original units": VETO*(BOUNDS["upper"] - BOUNDS["lower"])
})

,objective,weight,lower,upper,veto in original units
crashes avoided,max,0.30,0,20,7.0
pavement PCI,min,0.25,0,100,35.0
daily users,max,0.25,0,50000,17500.0
cost,min,0.20,2,12,3.5


### 2.2. Performance coding
---
For consistent orientation, code each performance on $[0,1]$, with larger values preferred:

$$ \large
g_j(a_i)=\begin{cases}
\dfrac{x_{ij}-L_j}{U_j-L_j}, & \text{if criterion }j\text{ is maximized}, \\
\dfrac{U_j-x_{ij}}{U_j-L_j}, & \text{if criterion }j\text{ is minimized}.
\end{cases}
$$

This linear coding is a modeling choice. The vetoes are expressed on these coded scales; they can differ by criterion. ELECTRE II does not require identical ranges when vetoes are specified in each criterion's own units.

In [4]:
assert (BOUNDS["upper"] > BOUNDS["lower"]).all()
assert df.ge(BOUNDS["lower"]).all().all()
assert df.le(BOUNDS["upper"]).all().all()

df_n = (df - BOUNDS["lower"])/(BOUNDS["upper"] - BOUNDS["lower"])

for col in df_n.columns:
    if OBJECTIVES[col] == "min":
        df_n[col] = 1 - df_n[col]

df_n.style.background_gradient(vmin=0, vmax=1)

,crashes avoided,pavement PCI,daily users,cost
North Avenue,0.800000,0.900000,0.900000,0.400000
School Corridor,0.900000,0.700000,0.700000,0.300000
Industrial Link,0.700000,0.600000,0.600000,0.600000
Market Street,0.750000,0.500000,0.500000,0.500000
South Connector,0.600000,0.400000,0.400000,0.700000


### 2.3. Concordance
---
For an ordered pair $(a,b)$, sum the weights of criteria on which $a$ is **at least as good as** $b$:

$$ \large
J^+(a,b)=\{j:g_j(a)\geq g_j(b)\}, \qquad
c(a,b)=\sum_{j\in J^+(a,b)}w_j
$$

Equal performances support both directions. These are **true criteria**: no indifference or preference thresholds are used. Rows represent $a$ and columns represent $b$ in all pairwise matrices.

In [5]:
CONCORDANCE_SETS = {}
df_c = pd.DataFrame(0.0, index=df.index, columns=df.index)

for a in df.index:
    for b in df.index:
        support = df_n.loc[a] >= df_n.loc[b]
        CONCORDANCE_SETS[a, b] = list(df.columns[support])
        df_c.loc[a, b] = WEIGHTS[support].sum()

df_sets = pd.DataFrame([
    [f'{a} -> {b}', ", ".join(CONCORDANCE_SETS[a, b])]
    for a in df.index for b in df.index if a != b
], columns=["comparison", "supporting criteria"]).set_index("comparison")

df_sets

,supporting criteria
comparison,
North Avenue -> School Corridor,"pavement PCI, daily users, cost"
North Avenue -> Industrial Link,"crashes avoided, pavement PCI, daily users"
North Avenue -> Market Street,"crashes avoided, pavement PCI, daily users"
North Avenue -> South Connector,"crashes avoided, pavement PCI, daily users"
School Corridor -> North Avenue,crashes avoided
School Corridor -> Industrial Link,"crashes avoided, pavement PCI, daily users"
School Corridor -> Market Street,"crashes avoided, pavement PCI, daily users"
School Corridor -> South Connector,"crashes avoided, pavement PCI, daily users"
Industrial Link -> North Avenue,cost


In [6]:
df_c.style.background_gradient(vmin=0, vmax=1)

,North Avenue,School Corridor,Industrial Link,Market Street,South Connector
North Avenue,1.000000,0.700000,0.800000,0.800000,0.800000
School Corridor,0.300000,1.000000,0.800000,0.800000,0.800000
Industrial Link,0.200000,0.200000,1.000000,0.700000,0.800000
Market Street,0.200000,0.200000,0.300000,1.000000,0.800000
South Connector,0.200000,0.200000,0.200000,0.200000,1.000000


### 2.4. Disadvantages and vetoes
---
An assertion must pass the **no-veto condition** for every criterion:

$$ \large
g_j(b)-g_j(a)\leq v_j, \qquad \forall j
$$

Display the largest positive disadvantage as a diagnostic:

$$ \large
d(a,b)=\max\left(0,\max_j[g_j(b)-g_j(a)]\right)
$$

Here all $v_j=0.35$, so no veto is equivalent to $d(a,b)\leq0.35$. The code still tests each criterion separately. Equality is accepted, with a small numerical tolerance.

In [7]:
df_d = pd.DataFrame(0.0, index=df.index, columns=df.index)
df_no_veto = pd.DataFrame(True, index=df.index, columns=df.index)

for a in df.index:
    for b in df.index:
        disadvantages = (df_n.loc[b] - df_n.loc[a]).clip(lower=0)
        df_d.loc[a, b] = disadvantages.max()
        df_no_veto.loc[a, b] = (disadvantages <= VETO + TOLERANCE).all()

df_d.style.background_gradient(vmin=0, vmax=1)

,North Avenue,School Corridor,Industrial Link,Market Street,South Connector
North Avenue,0.000000,0.100000,0.200000,0.100000,0.300000
School Corridor,0.200000,0.000000,0.300000,0.200000,0.400000
Industrial Link,0.300000,0.200000,0.000000,0.050000,0.100000
Market Street,0.400000,0.200000,0.100000,0.000000,0.200000
South Connector,0.500000,0.300000,0.200000,0.150000,0.000000


In [8]:
df_no_veto.astype(int)

,North Avenue,School Corridor,Industrial Link,Market Street,South Connector
North Avenue,1,1,1,1,1
School Corridor,1,1,1,1,0
Industrial Link,1,1,1,1,1
Market Street,0,1,1,1,1
South Connector,0,1,1,1,1


### 2.5. Strong and weak outranking relations
---
For $r=1,2$, use $s_1=0.75$ and $s_2=0.65$:

$$ \large
aS^r b \iff
\begin{cases}
c(a,b)\geq s_r, \\
c(a,b)\geq c(b,a), \\
g_j(b)-g_j(a)\leq v_j, & \forall j,
\end{cases}
\qquad a\ne b
$$

The same vetoes apply to both relations, and $S^1\subseteq S^2$. The reverse-concordance condition prevents accepting an assertion with less support than its reverse. A **weak-only** arc belongs to $S^2$ but not $S^1$.

In [9]:
df_balance = df_c >= df_c.T - TOLERANCE

df_strong = (df_c >= C_STRONG - TOLERANCE) & df_balance & df_no_veto
df_weak = (df_c >= C_WEAK - TOLERANCE) & df_balance & df_no_veto

# np.fill_diagonal(df_strong.values, False)
# np.fill_diagonal(df_weak.values, False)

for i in range(min(df_strong.shape)):
    df_strong.iloc[i, i] = False

for i in range(min(df_weak.shape)):
    df_weak.iloc[i, i] = False

assert not (df_strong & ~df_weak).to_numpy().any()

In [10]:
df_strong.astype(int)

,North Avenue,School Corridor,Industrial Link,Market Street,South Connector
North Avenue,0,0,1,1,1
School Corridor,0,0,1,1,0
Industrial Link,0,0,0,0,1
Market Street,0,0,0,0,1
South Connector,0,0,0,0,0


In [11]:
df_weak.astype(int)

,North Avenue,School Corridor,Industrial Link,Market Street,South Connector
North Avenue,0,1,1,1,1
School Corridor,0,0,1,1,0
Industrial Link,0,0,0,1,1
Market Street,0,0,0,0,1
South Connector,0,0,0,0,0


In [12]:
df_edges = pd.DataFrame([
    [a, b, "strong" if df_strong.loc[a, b] else "weak only"]
    for a in df.index for b in df.index if df_weak.loc[a, b]
], columns=["from", "to", "relation"])

df_edges

,from,to,relation
0,North Avenue,School Corridor,weak only
1,North Avenue,Industrial Link,strong
2,North Avenue,Market Street,strong
3,North Avenue,South Connector,strong
4,School Corridor,Industrial Link,strong
5,School Corridor,Market Street,strong
6,Industrial Link,Market Street,weak only
7,Industrial Link,South Connector,strong
8,Market Street,South Connector,strong


North Avenue has concordance **0.70** against School Corridor: its pavement urgency, users and cost support the assertion. Its greatest disadvantage is **0.10** for crashes avoided. Thus it outranks School **weakly**, but not strongly. Industrial Link similarly outranks Market Street only weakly.

School Corridor has concordance **0.80** against South Connector, but its extra BRL 4 million exceeds the cost veto of BRL 3.5 million. That **direct arc is blocked**.

### 2.6. Strong classes and initial layers
---
First group each **strong cycle** into a single class, retaining its members **ex aequo**. A strongly connected component contains alternatives mutually reachable by directed paths. Between two classes, keep an arc if any of their members has an arc to the other class.

The resulting graph is acyclic. Repeatedly remove all classes with **no incoming strong arc** to obtain the initial descending layers. The following helpers perform these graph operations; the ELECTRE calculations remain explicit above.

In [13]:
def condense(relation):
    _, labels = connected_components(
        relation.to_numpy(dtype=int), directed=True, connection="strong"
    )
    groups = {
        f'G{i + 1}': list(relation.index[labels == label])
        for i, label in enumerate(dict.fromkeys(labels))
    }
    grouped = pd.DataFrame(False, index=groups, columns=groups)

    for a, members_a in groups.items():
        for b, members_b in groups.items():
            if a != b:
                grouped.loc[a, b] = relation.loc[members_a, members_b].to_numpy().any()

    return groups, grouped


def source_layers(relation):
    remaining = list(relation.index)
    layers = []

    while remaining:
        current = relation.loc[remaining, remaining]
        layer = [a for a in remaining if not current[a].any()]
        if not layer:
            raise ValueError("Condense cycles before extracting layers.")
        layers.append(layer)
        remaining = [a for a in remaining if a not in layer]

    return layers


STRONG_GROUPS, df_strong_groups = condense(df_strong)

pd.Series(STRONG_GROUPS, name="members").to_frame()

,members
G1,[North Avenue]
G2,[School Corridor]
G3,[Industrial Link]
G4,[Market Street]
G5,[South Connector]


In [14]:
STRONG_LAYERS = source_layers(df_strong_groups)

pd.DataFrame({
    "strong layer": np.arange(len(STRONG_LAYERS)) + 1,
    "members": [
        [a for group in layer for a in STRONG_GROUPS[group]]
        for layer in STRONG_LAYERS
    ]
})

,strong layer,members
0,1,"[North Avenue, School Corridor]"
1,2,"[Industrial Link, Market Street]"
2,3,[South Connector]


### 2.7. Descending preorder
---
Refine **each strong layer separately** using weak arcs between its strong classes. Remove successive weak sources within that layer, finishing it before moving to the next strong layer.

Strong cycles remain tied. As an explicit implementation convention, any cycle encountered during **weak refinement** is also kept together as an unresolved tied block. Both examples are acyclic and do not depend on this convention.

In [15]:
df_weak_groups = pd.DataFrame(False, index=STRONG_GROUPS, columns=STRONG_GROUPS)

for a, members_a in STRONG_GROUPS.items():
    for b, members_b in STRONG_GROUPS.items():
        if a != b:
            df_weak_groups.loc[a, b] = df_weak.loc[members_a, members_b].to_numpy().any()


def refine_layers(strong_layers, weak_relation, strong_groups):
    result = []
    steps = []

    for strong_level, layer in enumerate(strong_layers, 1):
        weak_groups, weak_dag = condense(weak_relation.loc[layer, layer])
        for weak_level, batch in enumerate(source_layers(weak_dag), 1):
            selected = [group for node in batch for group in weak_groups[node]]
            members = [a for group in selected for a in strong_groups[group]]
            result.append(members)
            steps.append([strong_level, weak_level, members])

    trace = pd.DataFrame(steps, columns=["strong layer", "weak layer", "members"])
    return result, trace


DOWN, df_down = refine_layers(STRONG_LAYERS, df_weak_groups, STRONG_GROUPS)

df_down

,strong layer,weak layer,members
0,1,1,[North Avenue]
1,1,2,[School Corridor]
2,2,1,[Industrial Link]
3,2,2,[Market Street]
4,3,1,[South Connector]


### 2.8. Ascending preorder
---
Repeat the procedure from the **worst** end: remove classes with **no outgoing strong arc**, then refine each layer from its weak sinks. This is equivalent to applying the same helpers to the **transposed relations**.

The trace below follows extraction from worst to best. Reverse the **final list of layers** to present both preorders from best to worst, assigning rank **1** to the leading class.

In [16]:
UP_STRONG_LAYERS = source_layers(df_strong_groups.T)

UP_WORST_FIRST, df_up = refine_layers(
    UP_STRONG_LAYERS, df_weak_groups.T, STRONG_GROUPS
)

UP = UP_WORST_FIRST[::-1]

df_up

,strong layer,weak layer,members
0,1,1,[South Connector]
1,2,1,[Market Street]
2,2,2,[Industrial Link]
3,3,1,[School Corridor]
4,3,2,[North Avenue]


In [17]:
rank_down = pd.Series({
    a: level for level, layer in enumerate(DOWN, 1) for a in layer
})
rank_up = pd.Series({
    a: level for level, layer in enumerate(UP, 1) for a in layer
})

df_orders = pd.DataFrame({
    "descending": rank_down,
    "ascending": rank_up
}).reindex(df.index)

df_orders

,descending,ascending
North Avenue,1,1
School Corridor,2,2
Industrial Link,3,3
Market Street,4,4
South Connector,5,5


### 2.9. Intersection and final ranking
---
Intersect the two preorders, rather than averaging their ranks:

$$ \large
aZb \iff r_\downarrow(a)\leq r_\downarrow(b)
\quad\text{and}\quad
r_\uparrow(a)\leq r_\uparrow(b)
$$

For distinct alternatives, **P** means preferred, **P-** means the reverse preference, **I** means tied in both preorders, and **R** means incomparable. Incomparability occurs when the two directions disagree; it is different from a tie.

In [18]:
df_final = pd.DataFrame(False, index=df.index, columns=df.index)

for a in df.index:
    for b in df.index:
        df_final.loc[a, b] = (
            rank_down[a] <= rank_down[b] and rank_up[a] <= rank_up[b]
        )

df_pairwise = pd.DataFrame("R", index=df.index, columns=df.index)

for a in df.index:
    for b in df.index:
        if a == b:
            df_pairwise.loc[a, b] = "-"
        elif df_final.loc[a, b] and df_final.loc[b, a]:
            df_pairwise.loc[a, b] = "I"
        elif df_final.loc[a, b]:
            df_pairwise.loc[a, b] = "P"
        elif df_final.loc[b, a]:
            df_pairwise.loc[a, b] = "P-"

df_pairwise

,North Avenue,School Corridor,Industrial Link,Market Street,South Connector
North Avenue,-,P,P,P,P
School Corridor,P-,-,P,P,P
Industrial Link,P-,P-,-,P,P
Market Street,P-,P-,P-,-,P
South Connector,P-,P-,P-,P-,-


In [19]:
# Both preorders agree here, so the final order is complete and has no ties.
assert (df_orders["descending"] == df_orders["ascending"]).all()

df_rank = df.join(df_orders)
df_rank["ranking"] = df_orders["descending"]

df_rank.sort_values("ranking").style.background_gradient(subset=list(df.columns))

,crashes avoided,pavement PCI,daily users,cost,descending,ascending,ranking
North Avenue,16,10,45000,8,1,1,1
School Corridor,18,30,35000,9,2,2,2
Industrial Link,14,40,30000,6,3,3,3
Market Street,15,50,25000,7,4,4,4
South Connector,12,60,20000,5,5,5,5


The final priority is **North Avenue > School Corridor > Industrial Link > Market Street > South Connector**. North's broader supporting coalition takes precedence over School's larger crash reduction. Weak refinement separates North from School and Industrial from Market within their respective strong layers.

School ranks above South through the exploitation procedure, although their direct arc was vetoed. A final ordering is not a claim that every preferred pair has a direct outranking arc.

## 3. Examples
---

### 3.1. Municipal service digitalization
---
A municipality prioritizes **4 digital services** using **annual users**, **staff hours saved per year**, **implementation time** (months), and **cost** (BRL thousand). All meet the required accessibility and security requirements. The data and preferences are **illustrative**.

Use the same weights, concordance levels and coded vetoes as above, with the new planning bounds below. This example illustrates a clear first and last priority while preserving a meaningful **incomparable pair** in the middle.

In [20]:
df_ex1 = pd.DataFrame({
    "annual users": [16000, 14000, 18000, 10000],
    "staff hours saved": [4500, 4000, 3500, 3000],
    "implementation": [2, 4, 6, 8],
    "cost": [300, 260, 340, 220]
}, index=[
    "Case status alerts",
    "Appointment booking",
    "Online permit portal",
    "Archive search"
])

df_ex1.style.background_gradient()

,annual users,staff hours saved,implementation,cost
Case status alerts,16000,4500,2,300
Appointment booking,14000,4000,4,260
Online permit portal,18000,3500,6,340
Archive search,10000,3000,8,220


In [21]:
OBJECTIVES_ex1 = pd.Series(["max", "max", "min", "min"], index=df_ex1.columns)
WEIGHTS_ex1 = pd.Series([0.30, 0.25, 0.25, 0.20], index=df_ex1.columns)
VETO_ex1 = pd.Series(0.35, index=df_ex1.columns)
C_STRONG_ex1 = 0.75
C_WEAK_ex1 = 0.65

BOUNDS_ex1 = pd.DataFrame({
    "lower": [0, 0, 0, 100],
    "upper": [20000, 5000, 20, 500]
}, index=df_ex1.columns)

pd.DataFrame({
    "objective": OBJECTIVES_ex1,
    "weight": WEIGHTS_ex1,
    "lower": BOUNDS_ex1["lower"],
    "upper": BOUNDS_ex1["upper"],
    "veto in original units": VETO_ex1*(BOUNDS_ex1["upper"] - BOUNDS_ex1["lower"])
})

,objective,weight,lower,upper,veto in original units
annual users,max,0.30,0,20000,7000.0
staff hours saved,max,0.25,0,5000,1750.0
implementation,min,0.25,0,20,7.0
cost,min,0.20,100,500,140.0


In [22]:
# Performance coding
assert df_ex1.ge(BOUNDS_ex1["lower"]).all().all()
assert df_ex1.le(BOUNDS_ex1["upper"]).all().all()

df_ex1_n = (
    (df_ex1 - BOUNDS_ex1["lower"])
    /(BOUNDS_ex1["upper"] - BOUNDS_ex1["lower"])
)

for col in df_ex1_n.columns:
    if OBJECTIVES_ex1[col] == "min":
        df_ex1_n[col] = 1 - df_ex1_n[col]

df_ex1_n.style.background_gradient(vmin=0, vmax=1)

,annual users,staff hours saved,implementation,cost
Case status alerts,0.800000,0.900000,0.900000,0.500000
Appointment booking,0.700000,0.800000,0.800000,0.600000
Online permit portal,0.900000,0.700000,0.700000,0.400000
Archive search,0.500000,0.600000,0.600000,0.700000


In [23]:
# Concordance, largest disadvantages and criterion vetoes
df_ex1_c = pd.DataFrame(0.0, index=df_ex1.index, columns=df_ex1.index)
df_ex1_d = df_ex1_c.copy()
df_ex1_no_veto = pd.DataFrame(True, index=df_ex1.index, columns=df_ex1.index)

for a in df_ex1.index:
    for b in df_ex1.index:
        support = df_ex1_n.loc[a] >= df_ex1_n.loc[b]
        df_ex1_c.loc[a, b] = WEIGHTS_ex1[support].sum()
        disadvantages = (df_ex1_n.loc[b] - df_ex1_n.loc[a]).clip(lower=0)
        df_ex1_d.loc[a, b] = disadvantages.max()
        df_ex1_no_veto.loc[a, b] = (disadvantages <= VETO_ex1 + TOLERANCE).all()

pd.concat({"concordance": df_ex1_c, "disadvantage": df_ex1_d}, axis=1)

concordance                      \
                     Case status alerts Appointment booking   
Case status alerts                  1.0                 0.8   
Appointment booking                 0.2                 1.0   
Online permit portal                0.3                 0.3   
Archive search                      0.2                 0.2   

                                                               disadvantage  \
                     Online permit portal Archive search Case status alerts   
Case status alerts                    0.7            0.8                0.0   
Appointment booking                   0.7            0.8                0.1   
Online permit portal                  1.0            0.8                0.2   
Archive search                        0.2            1.0                0.3   

                                                                              
                     Appointment booking Online permit portal Archive search  
Case status alerts                   0.1                  0.1            0.2  
Appointment booking                  0.0                  0.2            0.1  
Online permit portal                 0.2                  0.0            0.3  
Archive search                       0.2                  0.4            0.0

In [24]:
# Strong and weak relations
df_ex1_balance = df_ex1_c >= df_ex1_c.T - TOLERANCE

df_ex1_strong = (
    (df_ex1_c >= C_STRONG_ex1 - TOLERANCE) & df_ex1_balance & df_ex1_no_veto
)
df_ex1_weak = (
    (df_ex1_c >= C_WEAK_ex1 - TOLERANCE) & df_ex1_balance & df_ex1_no_veto
)

for i in range(min(df_ex1_strong.shape)):
    df_ex1_strong.iloc[i, i] = False

for i in range(min(df_ex1_weak.shape)):
    df_ex1_weak.iloc[i, i] = False

pd.concat({"strong": df_ex1_strong.astype(int), "weak": df_ex1_weak.astype(int)}, axis=1)

strong                      \
                     Case status alerts Appointment booking   
Case status alerts                    0                   1   
Appointment booking                   0                   0   
Online permit portal                  0                   0   
Archive search                        0                   0   

                                                                       weak  \
                     Online permit portal Archive search Case status alerts   
Case status alerts                      0              1                  0   
Appointment booking                     0              1                  0   
Online permit portal                    0              1                  0   
Archive search                          0              0                  0   

                                                                              
                     Appointment booking Online permit portal Archive search  
Case status alerts                     1                    1              1  
Appointment booking                    0                    1              1  
Online permit portal                   0                    0              1  
Archive search                         0                    0              0

In [25]:
# Strong classes and the weak relation between classes
GROUPS_ex1, df_ex1_strong_groups = condense(df_ex1_strong)
df_ex1_weak_groups = pd.DataFrame(False, index=GROUPS_ex1, columns=GROUPS_ex1)

for a, members_a in GROUPS_ex1.items():
    for b, members_b in GROUPS_ex1.items():
        if a != b:
            df_ex1_weak_groups.loc[a, b] = df_ex1_weak.loc[members_a, members_b].to_numpy().any()

# Refine the strong layers separately in both directions
DOWN_ex1, df_ex1_down = refine_layers(
    source_layers(df_ex1_strong_groups), df_ex1_weak_groups, GROUPS_ex1
)
UP_ex1_worst_first, df_ex1_up = refine_layers(
    source_layers(df_ex1_strong_groups.T), df_ex1_weak_groups.T, GROUPS_ex1
)
UP_ex1 = UP_ex1_worst_first[::-1]

pd.concat({"descending extraction": df_ex1_down, "ascending extraction": df_ex1_up})

strong layer  weak layer                 members
descending extraction 0             1           1    [Case status alerts]
                      1             1           2  [Online permit portal]
                      2             2           1   [Appointment booking]
                      3             3           1        [Archive search]
ascending extraction  0             1           1        [Archive search]
                      1             2           1  [Online permit portal]
                      2             2           2   [Appointment booking]
                      3             3           1    [Case status alerts]

In [26]:
rank_down_ex1 = pd.Series({
    a: level for level, layer in enumerate(DOWN_ex1, 1) for a in layer
})
rank_up_ex1 = pd.Series({
    a: level for level, layer in enumerate(UP_ex1, 1) for a in layer
})

df_ex1_orders = pd.DataFrame({
    "descending": rank_down_ex1,
    "ascending": rank_up_ex1
}).reindex(df_ex1.index)

df_ex1_orders

,descending,ascending
Case status alerts,1,1
Appointment booking,3,2
Online permit portal,2,3
Archive search,4,4


In [27]:
# Intersect the preorders without assigning an artificial total rank.
df_ex1_final = pd.DataFrame(False, index=df_ex1.index, columns=df_ex1.index)

for a in df_ex1.index:
    for b in df_ex1.index:
        df_ex1_final.loc[a, b] = (
            rank_down_ex1[a] <= rank_down_ex1[b]
            and rank_up_ex1[a] <= rank_up_ex1[b]
        )

df_ex1_pairwise = pd.DataFrame("R", index=df_ex1.index, columns=df_ex1.index)

for a in df_ex1.index:
    for b in df_ex1.index:
        if a == b:
            df_ex1_pairwise.loc[a, b] = "-"
        elif df_ex1_final.loc[a, b] and df_ex1_final.loc[b, a]:
            df_ex1_pairwise.loc[a, b] = "I"
        elif df_ex1_final.loc[a, b]:
            df_ex1_pairwise.loc[a, b] = "P"
        elif df_ex1_final.loc[b, a]:
            df_ex1_pairwise.loc[a, b] = "P-"

df_ex1_pairwise

,Case status alerts,Appointment booking,Online permit portal,Archive search
Case status alerts,-,P,P,P
Appointment booking,P-,-,R,P
Online permit portal,P-,R,-,P
Archive search,P-,P-,P-,-


**Case status alerts** leads in both preorders, and **Archive search** is last. **Appointment booking** and **Online permit portal** are incomparable in the intersection: the descending preorder places the portal ahead of booking, while the ascending preorder places booking ahead of the portal.

In descending extraction, alerts and the portal share the first strong layer, ahead of booking's layer. In ascending extraction, booking and the portal share a strong layer, and their weak comparison favors booking. The trace tables show how this disagreement arises.

The portal reaches more users; booking saves more staff time, takes less time to implement and costs less. Their relation is **R**, rather than **I**. Averaging the two ranks would incorrectly turn this disagreement into a tie. The recommendation is therefore clear: prioritize alerts, retain the middle pair for further deliberation, and place archive search last.

## 4. References
---

- Figueira, J., Mousseau, V. and Roy, B. (2005). [*ELECTRE methods*](https://www.lamsade.dauphine.fr/mcda/biblio/PDF/JFVMBR2005.pdf). In *Multiple Criteria Decision Analysis: State of the Art Surveys*, pp. 133–162. Sections 3.1.2 and 3.2.1 specify the veto condition, two concordance levels and intersection procedure used here.
- Bouyssou, D. and Vanderpooten, D. (2011). [*Bernard Roy*](https://www.lamsade.dauphine.fr/~bouyssou/Bouyssou-Vanderpooten2011_Chapter_BernardRoy.pdf). In *Profiles in Operations Research*. Describes ELECTRE II as a ranking method that preserves incomparability.